Вот конвертация HTML → Markdown:

---

Notebooks с семинара можно найти по [ссылке 1](https://colab.research.google.com/github/huggingface/cookbook/blob/main/notebooks/en/multimodal_rag_using_document_retrieval_and_vlms.ipynb), [ссылке 2](https://colab.research.google.com/drive/1J1BdJRB4EEpmWBrW5t9cDK-0yD-DEIBP?usp=sharing).

### Домашнее задание 12

**1. Часть 1 (5 баллов):**

* Заполнить дизайн-документ для RAG (см. [шаблон](https://colab.research.google.com/drive/1KBG74ca-fLKqORUZxfLiIQiNP1FvH0ZN?usp=sharing))

**2. Часть 2 (5 баллов):**

*На выбор два варианта:*

* Попробовать завести графовый RAG (FastRAG, Light-RAG, mini-RAG) на ранее собранных данных, замерить результаты ретрива и протестировать генерацию
* Попробовать завести мультимодальный RAG на любого рода картинках, инструкциях и протестировать на собранных запросах

> В обоих случаях объяснить выбор метода подсчёта метрик и предоставить эксперименты (опробовать различные библиотеки, модели, промпты).

**Домашнее задание необходимо предоставить в формате ссылки на Google Collab/Jupiter Notebook с вашими действиями и ключевыми выводами:**

* 24 ноября 23:59 — мягкий дедлайн
* 1 декабря 23:59 — жесткий дедлайн

До мягкого дедлайна за работу можно получить 10 баллов, после — 5; работы, отправленные после 1 декабря, могут быть проверены преподавателями до конца курса в формате зачет/не зачет.


**1. Часть 1 (5 баллов):**

* Заполнить дизайн-документ для RAG (см. [шаблон](https://colab.research.google.com/drive/1KBG74ca-fLKqORUZxfLiIQiNP1FvH0ZN?usp=sharing))


**2. Часть 2 (5 баллов):**

*На выбор два варианта:*

* Попробовать завести графовый RAG (FastRAG, Light-RAG, mini-RAG) на ранее собранных данных, замерить результаты ретрива и протестировать генерацию
* Попробовать завести мультимодальный RAG на любого рода картинках, инструкциях и протестировать на собранных запросах

> В обоих случаях объяснить выбор метода подсчёта метрик и предоставить эксперименты (опробовать различные библиотеки, модели, промпты).

In [1]:
!pip uninstall -y langchain langchain-core langchain-text-splitters langchain-qdrant langchain-huggingface
!pip uninstall -y qdrant-client sentence-transformers transformers huggingface-hub
!pip uninstall qdrant-client

Found existing installation: langchain 1.0.0
Uninstalling langchain-1.0.0:
  Successfully uninstalled langchain-1.0.0
Found existing installation: langchain-core 1.1.0
Uninstalling langchain-core-1.1.0:
  Successfully uninstalled langchain-core-1.1.0
Found existing installation: langchain-text-splitters 1.0.0
Uninstalling langchain-text-splitters-1.0.0:
  Successfully uninstalled langchain-text-splitters-1.0.0
Found existing installation: langchain-qdrant 1.0.0
Uninstalling langchain-qdrant-1.0.0:
  Successfully uninstalled langchain-qdrant-1.0.0
Found existing installation: langchain-huggingface 1.0.1
Uninstalling langchain-huggingface-1.0.1:
  Successfully uninstalled langchain-huggingface-1.0.1
Found existing installation: qdrant-client 1.16.0
Uninstalling qdrant-client-1.16.0:
  Successfully uninstalled qdrant-client-1.16.0
Found existing installation: sentence-transformers 2.6.0
Uninstalling sentence-transformers-2.6.0:
  Successfully uninstalled sentence-transformers-2.6.0
Found 

In [2]:
!pip install \
  tqdm \
  pandas \
  langchain==1.0.0 \
  langchain-huggingface==1.0.1 \
  langchain-qdrant==1.0.0 \
  langchain-text-splitters==1.0.0 \
  qdrant-client==1.16.0 \
  huggingface_hub==0.34.0 \
  sentence-transformers==2.6.0 \
  transformers==4.40.0 \
  fastembed


  Using cached langchain-1.0.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached langchain_huggingface-1.0.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached langchain_qdrant-1.0.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached langchain_text_splitters-1.0.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached qdrant_client-1.16.0-py3-none-any.whl.metadata (11 kB)
  Using cached huggingface_hub-0.34.0-py3-none-any.whl.metadata (14 kB)
  Using cached sentence_transformers-2.6.0-py3-none-any.whl.metadata (11 kB)
  Using cached transformers-4.40.0-py3-none-any.whl.metadata (137 kB)
  Using cached langchain_core-1.1.0-py3-none-any.whl.metadata (3.6 kB)
Using cached langchain-1.0.0-py3-none-any.whl (106 kB)
Using cached langchain_huggingface-1.0.1-py3-none-any.whl (27 kB)
Using cached huggingface_hub-0.34.0-py3-none-any.whl (558 kB)
Using cached langchain_qdrant-1.0.0-py3-none-any.whl (24 kB)
Using cached qdrant_client-1.16.0-py3-none-any.whl (328 kB)
Using cached langchain_text_splitters-

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import Qdrant
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document

/Users/sergey/Projects/GigaSchool/llm-engineer/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Step 1 — Text cleaning (procedural)
# ----------------------------------
# Usage: set `csv_path` to your CSV file, run the cell. It will produce `data_cleaned.csv`.

import re
import pandas as pd

# emoji regex (covers common ranges)
_emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map symbols
    "\U0001F1E0-\U0001F1FF"  # flags
    "]+",
    flags=re.UNICODE
)

def clean_text_proc(text: str) -> str:
    """Clean a single text string: remove URLs, markdown links, hashtags (keep words), emojis, and extra whitespace."""
    if not isinstance(text, str):
        return ""
    # remove URLs
    text = re.sub(r'https?://\S+', '', text)
    # convert markdown links [label](url) -> label
    text = re.sub(r'\[(.*?)\]\(.*?\)', lambda m: m.group(1), text)
    # remove leading/trailing hashes while keeping the word
    text = re.sub(r'#(\w+)', lambda m: m.group(1), text)
    # strip emojis
    text = _emoji_pattern.sub('', text)
    # collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Procedural flow
csv_path = 'data/channel_data.csv' 
cleaned_path = 'data/channel_data_cleaned.csv'

print('Loading', csv_path)
df = pd.read_csv(csv_path, parse_dates=['date'])
print('Rows loaded:', len(df))

print('Cleaning text...')
df['text_clean'] = df['text'].apply(clean_text_proc)

print('Sample cleaned texts:')
print(df[['id', 'text_clean']].head().to_string())

print('Saving cleaned CSV to', cleaned_path)
df.to_csv(cleaned_path, index=False)
print('Done.')


Loading data/channel_data.csv
Rows loaded: 1144
Cleaning text...
Sample cleaned texts:
     id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                text_clean
0  1201                                                                                                                                                                                        

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)


doc_list = text_splitter.split_documents([
    Document(
        page_content=row['text'],
        metadata={
            "title": row['id']
            }
        )
        for i, row in df.iterrows()
    ])
doc_list

[Document(metadata={'title': 1201}, page_content='До Стачки осталось совсем немного времени. Подробнее об этой конференции я писал в'),
 Document(metadata={'title': 1201}, page_content='я писал в [посте](https://t.me/ohmyflutter/1195), и мы даже успели разыграть пару билетов.'),
 Document(metadata={'title': 1201}, page_content='У тех, кому в розыгрыше не повезло, но поучаствовать хочется, все еще есть время даже до “late'),
 Document(metadata={'title': 1201}, page_content='время даже до “late bird” цен.'),
 Document(metadata={'title': 1201}, page_content='📌 Программа и билеты доступны по ссылке.\nhttps://spb25.nastachku.ru/\n\n#event'),
 Document(metadata={'title': 1198}, page_content='FlutterCon Europe 25 начался 🚀🚀🚀'),
 Document(metadata={'title': 1198}, page_content='Кстати организаторы обещали в этот раз появление докладов у них на ютуб канале прямо в день'),
 Document(metadata={'title': 1198}, page_content='канале прямо в день выступления.'),
 Document(metadata={'title': 1197}, pa

In [6]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")


print(device)

mps


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

# symetric model
embed_model_name = "BAAI/bge-m3"
embed_model_kwargs = {'device': device}
embed_encode_kwargs = {'normalize_embeddings': True}
query_encode_kwargs = {}

# asymetric model
"""
embed_model_name = "intfloat/multilingual-e5-large"
embed_model_kwargs = {'device': device}
encode_kwargs = {"prompt": "passage: "} # prompts
query_encode_kwargs = {"prompt": "query: "} # prompts
"""

embeddings = HuggingFaceEmbeddings(
    model_name=embed_model_name,
    model_kwargs=embed_model_kwargs,
    encode_kwargs=embed_encode_kwargs,
)

/Users/sergey/Projects/GigaSchool/llm-engineer/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/sergey/Projects/GigaSchool/llm-engineer/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
import pkg_resources

print("qdrant-client:", pkg_resources.get_distribution("qdrant-client").version)
print("langchain-qdrant:", pkg_resources.get_distribution("langchain-qdrant").version)
print("langchain:", pkg_resources.get_distribution("langchain").version)


qdrant-client: 1.16.0
langchain-qdrant: 1.0.0
langchain: 1.0.0


/var/folders/38/2h99w6rx1fz1qhgr17m1cbpm0000gn/T/ipykernel_49498/84277223.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [9]:
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")
qdrant = QdrantVectorStore.from_documents(
    doc_list,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    location=":memory:",
    collection_name="my_documents_2",
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="custom_vector",
    sparse_vector_name="custom_sparse_vector",
)


qdrant_retriever = qdrant.as_retriever(verbose=True)

In [10]:
qdrant_retriever.invoke("Как стать Flutter разработчиком?")

[Document(metadata={'title': 87, '_id': 'df344e5157744e6abe67c6379dd4cbc1', '_collection_name': 'my_documents_2'}, page_content='📌Нужно ли изучать Flutter и где искать работу Flutter-разработчику'),
 Document(metadata={'title': 87, '_id': 'b5273d5f028a4c8384f2c33f21dacfd2', '_collection_name': 'my_documents_2'}, page_content='Flutter или стать Flutter-разработчиком, чтобы больше зарабатывать.'),
 Document(metadata={'title': 1152, '_id': '76011cbce303465793be977d41c54895', '_collection_name': 'my_documents_2'}, page_content='Компания Founders ищет разработчика на фреймворке Flutter'),
 Document(metadata={'title': 1055, '_id': 'f9810f6834734fffb1360daeb3d66e17', '_collection_name': 'my_documents_2'}, page_content='Также как и [drawRawAtlas](https://api.flutter.dev/flutter/dart-ui/Canvas/drawRawAtlas.html)')]

In [11]:
import random

def generate_eval_set_from_posts(df, n_samples=5):
    """
    Генерирует eval set из DataFrame с колонкой 'text'.
    Возвращает список словарей: 
    { 'query': ..., 'expected_answer': ..., 'keywords': [...] }
    """
    eval_set = []
    
    # случайные n_samples постов
    sample_posts = df.sample(min(n_samples, len(df)), random_state=42)
    
    for _, row in sample_posts.iterrows():
        text = row["text"]
        
        # 1. Создаем "вопрос" для теста
        # Берем первые 1-2 предложения как основу вопроса
        sentences = text.split("\n")
        first_sentence = sentences[0] if sentences else text
        query = f"Что говорится в посте про: '{first_sentence[:50]}...'?".strip()
        
        # 2. Берем ключевые слова из текста
        # Например, все заглавные слова / имена собственные как ключевые слова
        import re
        keywords = re.findall(r'\b[A-ZА-Я][a-zа-яA-Z]+\b', text)
        keywords = list(set(keywords))  # уникальные
        
        # Если ключевых слов мало, добавляем первые 5 слов текста
        if len(keywords) < 3:
            keywords.extend(text.split()[:5])
            keywords = list(set(keywords))
        
        eval_set.append({
            "query": query,
            "expected_answer": text[:100],  # первые 100 символов как эталон
            "keywords": keywords
        })
    
    return eval_set

# Пример использования
eval_set = generate_eval_set_from_posts(df, n_samples=5)
for e in eval_set:
    print("\nQuery:", e["query"])
    print("Keywords:", e["keywords"])



Query: Что говорится в посте про: 'Всем привет! Я тут пытаюсь добавить Elementary в с...'?
Keywords: ['Elementary', 'Всем', 'PR', 'Спасибо']

Query: Что говорится в посте про: 'Даже самое классное и качественно реализованное пр...'?
Keywords: ['Во', 'Как', 'Owen', 'Даже', 'Flutter', 'Если', 'Tony']

Query: Что говорится в посте про: 'Двадцать лет назад компания Google запустила глоба...'?
Keywords: ['Двадцать', 'Хотите', 'Можно', 'Google', 'Code', 'Jam']

Query: Что говорится в посте про: 'Международная академия EDPRO ищет опытного flutter...'?
Keywords: ['Подробности', 'Международная', 'Контакт', 'Обязанности', 'NDA', 'Обязательно', 'Заработная', 'EDPRO']

Query: Что говорится в посте про: 'А тем временем, в одной из ближайших версий, канет...'?
Keywords: ['Подробности', 'одной', 'А', 'тем', 'временем,', 'в']


In [39]:
async def qdrant_retrieve(query, top_k=5):
    retriever = qdrant.as_retriever(search_kwargs={"k": top_k})
    docs = await retriever.ainvoke(query)
    return [doc.page_content for doc in docs]


In [40]:
import time
import pandas as pd

async def evaluate_qdrant(eval_set):
    results = []
    for row in eval_set:
        query = row["query"]
        keywords = [k.lower() for k in row["keywords"]]

        t0 = time.time()
        retrieved = await qdrant_retrieve(query, top_k=5)  # ждем результат
        latency = time.time() - t0

        retrieved_lower = [r.lower() for r in retrieved]
        hits = sum(any(k in r for r in retrieved_lower) for k in keywords)
        precision = hits / len(retrieved) if retrieved else 0
        recall = hits / len(keywords) if keywords else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        results.append({
            "query": query,
            "latency": latency,
            "hits": hits,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

    return pd.DataFrame(results)


In [41]:
metrics_df = await evaluate_qdrant(eval_set)
print(metrics_df.describe())


        latency      hits  precision    recall        f1
count  5.000000  5.000000   5.000000  5.000000  5.000000
mean   0.106260  2.600000   0.520000  0.445238  0.474670
std    0.044831  1.516575   0.303315  0.267473  0.281688
min    0.083512  1.000000   0.200000  0.142857  0.166667
25%    0.085453  2.000000   0.400000  0.250000  0.307692
50%    0.087773  2.000000   0.400000  0.500000  0.444444
75%    0.088175  3.000000   0.600000  0.500000  0.545455
max    0.186386  5.000000   1.000000  0.833333  0.909091


In [42]:
metrics_df

,query,latency,hits,precision,recall,f1
0,Что говорится в посте про: 'Всем привет! Я тут...,0.186386,2,0.4,0.500000,0.444444
1,Что говорится в посте про: 'Даже самое классно...,0.088175,1,0.2,0.142857,0.166667
2,Что говорится в посте про: 'Двадцать лет назад...,0.087773,3,0.6,0.500000,0.545455
3,Что говорится в посте про: 'Международная акад...,0.083512,2,0.4,0.250000,0.307692
4,"Что говорится в посте про: 'А тем временем, в ...",0.085453,5,1.0,0.833333,0.909091



## 📊 Метрики Qdrant

```
        latency      hits  precision    recall        f1  
count   5.000      5.000   5.000        5.000         5.000  
mean    0.1063     2.6     0.52         0.4452        0.4747  
std     0.0448     1.5166  0.3033       0.2675        0.2817  
min     0.0835     1.0     0.20         0.1429        0.1667  
25%     0.0855     2.0     0.40         0.25          0.3077  
50%     0.0878     2.0     0.40         0.50          0.4444  
75%     0.0882     3.0     0.60         0.50          0.5455  
max     0.1864     5.0     1.00         0.8333        0.9091  
```

---

## 🔍 Анализ

1. **Latency**

   * В среднем поиск через Qdrant занимает примерно **0.106 сек**, что очень быстро.
   * Разброс по времени небольшой (стандартное отклонение ≈ 0.045 сек), минимальное время почти стандардное — 0.083 сек, максимум — около 0.186 сек. Это очень хороший уровень для векторной базы: высокая скорость.

2. **Hits (попадания ключевых слов)**

   * В среднем `hits ≈ 2.6` — значит в среднем в возвращённых топ-к результатах содержатся ~2.6 ключевых слов из eval‑набора.
   * Большой диапазон: от 1 до 5 — значит, что в некоторых запросах retriever “видит” много релевантных сущностей, а в других — только одну.

3. **Precision (точность)**

   * Средняя точность `0.52` (52%) — очень достойный показатель для retrieval: больше половины найденных текстов содержали ожидаемые ключевые слова.
   * Разброс большой (стд ≈ 0.30): некоторые запросы дают очень высокую точность (до 1.0), а некоторые — низкую (0.20).

4. **Recall (полнота)**

   * Средний `recall ≈ 0.445` — почти половина эталонных ключевых слов находится в retrieved текстах.
   * Диапазон от ~0.14 до 0.83: для некоторых запросов retriever находит почти все ключевые слова, для других — лишь часть.

5. **F1-score**

   * Средний F1 ≈ 0.475 — хороший баланс между precision и recall на наборе.
   * Максимальный F1 — ~0.909, минимальный — ~0.167. Это говорит, что для некоторых запросов Qdrant работает очень хорошо, для некоторых — хуже.

---

## ✅ Вывод о Qdrant в RAG системе

* **Скорость:** Qdrant очень быстрый — latency ~0.1 сек делает его отличным кандидатом для низко-задержечного retrieval.
* **Качество retrieval:** В среднем retriever показывает **высокую точность (0.52)** и **среднюю полноту (0.445)**, что говорит, что он хорошо возвращает релевантные документы.
* **Надёжность:** Поскольку разброс по precision и recall значительный, некоторые запросы будут работать отлично, другие — с меньшим покрытием. Это нормально для семантического поиска.

